<a href="https://colab.research.google.com/github/SaurabhDangi/Celebal-Technology-tasks/blob/main/week7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
pip install langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu pypdf huggingface_hub

In [16]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFaceEndpoint
from langchain_classic.chains import RetrievalQA

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "your_hugging_face_token_here"

def build_and_run_rag(pdf_path, user_query):
    print("1. Document Ingestion: Loading the PDF...")
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    print("2. Text Chunking: Splitting text into smaller chunks...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    text_chunks = text_splitter.split_documents(documents)

    print("3. Embedding Creation & Vector Database: Storing embeddings...")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_database = FAISS.from_documents(text_chunks, embeddings)

    print("4 & 5. Query Processing & Context Retrieval: Setting up the retriever...")
    retriever = vector_database.as_retriever(search_kwargs={"k": 3})

    print("6. Answer Generation: Loading Language Model...\n")
    llm = HuggingFaceEndpoint(
        repo_id="mistralai/Mistral-7B-Instruct-v0.2",
        temperature=0.1,
        max_new_tokens=250
    )

    rag_pipeline = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )

    print(f"User Question: '{user_query}'")
    print("Generating Answer...\n")

    result = rag_pipeline.invoke({"query": user_query})

    print("--- ANSWER ---")
    print(result['result'])
    print("\n--- RETRIEVED CONTEXT (Sources) ---")
    for i, doc in enumerate(result['source_documents']):
        print(f"\nSource {i+1}:")
        print(doc.page_content)

if __name__ == "__main__":
    my_pdf = "sample.pdf"
    question = "What is the main idea of the document?"

    if os.path.exists(my_pdf):
        build_and_run_rag(my_pdf, question)
    else:
        print(f"Error: Please place a PDF named '{my_pdf}' in the same directory.")

Error: Please place a PDF named 'sample.pdf' in the same directory.
